# Charts: sampling coordinates vs. the timing model

Two different objects in this notebook are called a *chart*.

A **per-axis prior chart** maps a physical offset δ to a prior-normal `z`. Nothing is renamed; only the sampler's yardstick changes.

A **physical chart** changes *which parameters the sampler sees*. The engine's delay model is untouched.

## Why a binary orbit needs one

In a nearly circular binary the periastron is barely defined. `OM` is then poorly measured, and `T0` — the epoch of periastron passage — inherits that uncertainty: swinging `OM` through 360° slides `T0` by a whole `PB`. So `ECC`, `OM` and `T0` come out as a thin curved tube pressed up against `ECC = 0`. Steppers crawl along it and gradient samplers stall.

Timers already know the cure: fit `EPS1 = ECC·sin(OM)`, `EPS2 = ECC·cos(OM)` and `TASC`, the ascending-node epoch. Those are nearly uncorrelated and nearly Gaussian.

The new part is that **you do not have to change the binary model to get that geometry.** Setting `BINARY ELL1` in a par file also truncates the delay in powers of `ECC`; this chart does not. The engine keeps computing the full `DD`/`DDH` delay — same likelihood, same physics — while the priors and the sampler move to the ELL1-style coordinates. In the API the map is the `KeplerLaplace` chart: *Kepler* for the periastron set `ECC/OM/T0`, *Laplace* for the eccentricity-vector set `EPS1/EPS2/TASC`.

One thing genuinely does change: the prior. `dEPS1 dEPS2 = ECC·dECC·dOM`, so a flat `EPS` box means `p(ECC) ∝ ECC`, not uniform. Priors live in whichever frame is sampled, so chart-on and chart-off evidences only compare if you hold the prior fixed (taking into account the Jacobian).

Pulsar: AEI-DR2 combined J1022+1001 (`BINARY DDH`, `PB` = 7.8 d, `ECC` ≈ 1e-4). Its par-file `PX` is negative, so expand at the prior center rather than the par value.


In [ ]:
import os
os.environ.setdefault("JAX_ENABLE_X64", "1")  # timing residuals need float64

from pathlib import Path
from metapulsar import create_metapulsar
from nltiming import TimingSpec, TimingExpansionSpec
import nltiming.sampling as nlts

nlts.numpyro.ensure_x64()

# DDH binary, PB = 7.8 d, ECC ~ 1e-4 — the near-circular regime this notebook
# is about — with OMDOT, PBDOT and XDOT all free in the par file.
DATA = Path("..") / "data" / "J1022+1001"
pulsar = create_metapulsar(
    {"combined": [{
        "par": DATA / "J1022+1001.par",
        "tim": DATA / "J1022+1001.tim",
        "timing_package": "tempo2",
    }]},
    combination_strategy="per_pta",
    use_pulse_numbers="reuse",
)
print(pulsar.name, len(pulsar.toas))


## What `binary_chart="auto"` decides

Under `auto` the chart is a request, not a guarantee. When a check fails the chart is **demoted**: nothing is renamed, the plan keeps `ECC`, `OM` and `T0`, and you get exactly what `binary_chart="off"` would have given you, plus a one-line `UserWarning` and an `enabled: false` record carrying a `reason` code.

Demotion is a statement about coordinates, not about correctness. The posterior is the same distribution either way; you just explore it in the badly conditioned coordinates from the previous section. Expect the curved `ECC`–`OM`–`T0` tube, a step size pinned by its narrowest direction, and a far lower effective sample size per unit of wall time. Nothing is biased — the run is simply expensive.

Two checks account for nearly all demotions, and they fail for quite different reasons.

**The eccentricity origin.** `EPS1`/`EPS2` are Cartesian, so the *chart* is perfectly smooth through `ECC = 0` — removing that singularity is the entire point. The timing package, though, still evaluates the Roemer delay in `ECC`/`OM`. Tempo2 and PINT both have to convert those back internally, and some of the usual DD/`DDH` terms contain factors of `1/ECC` or `arctan2(EPS1, EPS2)`. Whether a given package (and the adapter that wraps it) stays finite and differentiable as `ECC → 0` is a numerical property of that implementation, not something the chart can promise.

**Certified** here means a recorded test, not a theorem. A timing-package adapter is `origin_certified` only after a full-likelihood run through `ECC = 0` has been shown to stay finite (no `1/ECC` blow-up), to keep HMC leapfrog stable, and to agree between Discovery and Enterprise. That flag is off until the PR that lands the passing run. No production adapter has that run yet, so today `auto` treats every timing package as uncertified.

An uncertified package is still fine *as long as the sampler is never allowed to visit `ECC = 0`*. That is a statement about the prior support. Picture `(EPS1, EPS2)` as a point in a plane; `ECC = 0` is the origin of that plane. The default prior is a box of half-width `50σ` centered on the par-file / WLS eccentricity vector (`nonlinear_scale`, σ from the par-file uncertainty or a WLS fit). For a typical MSP, `ECC` is only a few σ from zero, so the box is larger than the distance from the fitted point to the origin — the origin sits *inside* the allowed region. The sampler could then propose a circular orbit, the uncertified timing package would have to evaluate the delay there, and `auto` refuses the chart rather than take that risk. Narrowing the box so it no longer contains the origin (next section) is what lifts the check.

**The ω seam.** `TASC = T0 − PB·OM/2π` is an exact identity only for an orbit with no secular evolution. With `OMDOT` or `PBDOT` free it drifts, and the branch cut in `OM` then carries a real step of order `rate × PB` in the likelihood. That is a discontinuity rather than curvature, and it is fatal in a different way: a gradient sampler integrating across it reads a derivative that does not describe the function it is sampling. So when the epoch shift is inexact, the chart engages only if the support provably avoids that ray.

J1022+1001 has `OMDOT` and `PBDOT` free *and* a default box large enough to include `ECC = 0`, so both checks fail here. Read the manifest rather than guessing — the codes you are likely to see:

| `reason` | meaning |
|---|---|
| `seam_reachable_with_secular_terms` | inexact epoch shift, and the support reaches the ω branch cut |
| `origin_uncertified_backend` | the prior allows `ECC = 0`, and this timing package has not passed the origin test |
| `no_sampled_axis` | `ECC`, `OM` and `T0` are all marginalized (silent under `auto`) |
| `prior_on_kepler_axis` | a prior was declared on `ECC`, `OM` or `T0` |
| `e_ref_above_e_max` | reference `ECC` exceeds `e_max` (0.1 by default) |

That last one is policy, not a limit of the map: the chart is exact for every `ECC > 0`, but a genuinely eccentric binary has a well-defined periastron and does not need it.


In [ ]:
spec = TimingSpec(
    engines="jug", binary_chart="auto", name="timing",
    # Linearize at the center of each prior instead of the par-file value:
    # this par's PX is negative, which is not a usable parallax expansion point.
    expansion=TimingExpansionSpec.prior_center(),
)
timing = spec.for_pulsar(pulsar)

print(timing.plan)     # disposition per fitpar: sampled vs marginalized
print(timing.sampled)  # if the chart engaged, EPS1/EPS2/TASC show up here
                       # in place of ECC/OM/T0

# The audit trail: one record per orbital group, with `enabled` and, when the
# chart stood down, the `reason` code.
print(timing.binary_chart_manifest())

# Per-axis view. prior_chart is the delta -> z map (affine_normal for a Gaussian
# prior, prior_pit for anything else); physical_chart names the coordinate change
# if there is one; engine_name is the fitpar the delay model still works in.
for d in timing.chart_summary():
    print(d["name"], d["disposition"], d["prior_chart"], d["physical_chart"], d["engine_name"])


## Getting the chart to engage

Both guards test the *support of the prior*, so that is the only thing that can engage the chart — switching modes will not. `"auto"` and `"on"` differ only in what they do when a guard fails: `"auto"` warns and carries on in `ECC/OM/T0`, `"on"` raises. Use `"on"` in a pipeline, where a quiet fallback would waste a long run, and `"auto"` while you are still looking around.

It helps to picture `(EPS1, EPS2)` as a vector in a plane: its length is `ECC` and its direction is `OM`. The prior support is a box in that plane, and each guard is asking where the box sits.

- The **origin** of the plane is `ECC = 0`. Giving `EPS1` a lower bound of `1e-6` rather than `0` lifts the box clear of it.
- The **seam** is the ray leaving the origin in the direction exactly opposite today's eccentricity vector — the point where `OM` has swung a full 180° and the unwrapping branch flips. A box that stays on the near side of the origin never reaches it.

That reframes the question, but it does not make it free. A prior is a physical claim: `EPS1 > 1e-6` says you are confident the orbit is not circular, and a narrow box says you know `OM` to within a sector. If you believe both, the better geometry costs you nothing. If you shrank the box only to make `enabled` flip, you have bought sampling behaviour with an assumption you cannot defend — and the posterior will never warn you, because it cannot go where the prior forbids.

(A chart also needs at least one of `ECC`/`OM`/`T0` to be *sampled*. If all three are marginalized there is nothing to reparameterize, and the manifest records that quietly — see the last cell.)


In [ ]:
from nltiming.priors import uniform
from nltiming import laplace_from_kepler, kepler_from_laplace

spec_kl = TimingSpec(
    engines="jug",
    binary_chart="auto",
    # Boxes on the order of the WLS scale for ECC ~ 1e-4. EPS1 starts at 1e-6,
    # not 0: the support must not contain the eccentricity origin. Declaring
    # priors on EPS means the chart is in charge of that pair — a prior left on
    # ECC, OM or T0 would instead demote the chart, since a T0 density does not
    # carry over to TASC.
    priors={"EPS1": uniform(1e-6, 3e-4), "EPS2": uniform(-1e-4, 1e-4)},
    name="timing",
    expansion=TimingExpansionSpec.prior_center(),
)
timing_kl = spec_kl.for_pulsar(pulsar)
print(timing_kl.sampled)     # what the sampler moves in
print(timing_kl.delay_keys)  # what gets handed to the delay model
print(timing_kl.binary_chart_manifest())
for d in timing_kl.chart_summary():
    print(d["name"], d["prior_chart"], d["physical_chart"], d["engine_name"])

# The map itself, on this pulsar's par values: (ECC, OM in deg, T0, PB).
# TASC lands ~2.1 d before T0 — that offset is PB * OM / 360.
print(laplace_from_kepler(9.73e-5, 97.69, 50246.72, 7.805))
# Invertible: the round trip returns the periastron set. PB is needed both ways,
# because it sets the T0 <-> TASC epoch shift.
print(kepler_from_laplace(*laplace_from_kepler(9.73e-5, 97.69, 50246.72, 7.805), 7.805))


## What the sampler's parameter list looks like

Enterprise samples the prior-normal `z`, one scalar per axis, so the chart's whole visible footprint on the sampler side is a rename: `..._timing_ECC` becomes `..._timing_EPS1`. Same pulsar, same delay model, same number of parameters.


In [ ]:
from enterprise.signals import parameter, signal_base, white_signals

white = white_signals.MeasurementNoise(efac=parameter.Constant(1.0))
# Default spec: engine-frame orbital names.
print(signal_base.PTA([(white + spec.enterprise_signal())(pulsar)]).param_names)
# EPS-prior spec: the charted axes appear under their sampling-frame names.
print(signal_base.PTA([(white + spec_kl.enterprise_signal())(pulsar)]).param_names)


## Reporting in the usual parameters

Papers quote `ECC`, `OM` and `T0`, so the sampling frame is an internal detail. `derived_kepler_columns` adds those columns back from the `EPS1/EPS2/TASC` draws, decoding each one on the same reference-local ω branch the likelihood used — not a global `[0, 360)` wrap, which would shift `T0` by whole orbits for draws near the seam.

Derived columns appear only for groups whose charted axes were all sampled. A partly marginalized triple gets none, rather than columns half-built from reference values.

In [ ]:
from nltiming import derived_kepler_columns

# Takes any dict of sampling-frame columns (e.g. from nlts.numpyro.posterior()
# or a loaded run) and returns the ECC/OM/T0 columns to merge alongside:
#
#   post.update(derived_kepler_columns(post, timing_kl.binary_chart_manifest()))
#
# RunResults.posterior() already does this for you on a run written to disk.


## Where this goes next

The eccentricity vector is one physical chart; `fw10_absorbed` does the same job for the curved DDH `(A1, ECC, OM, T0, H3, STIGMA)` valley, and stays off here because of the secular dots and `STIG > 1`. And when an orbital triple is fully marginalized, so nothing is renamed at all, `marginal_basis_frame="auto"` still reconditions those design-matrix columns with the same eccentricity-vector geometry — the conditioning win does not depend on sampling the orbit.

Guard details, prior semantics and the `STIGMA` prior helpers are in the package README. Notebook 4 checks that the geometry you end up with is actually sampleable.
